# Run and monitor `run_exp_4.sh` from a notebook

**Where this lives:** put this file in `mh-dpo-master/` on Pegasus, next to `run_exp_4.sh`, and open it
with a Jupyter server running **on the GPU node** (so it can see the same GPU and the same `venv_x86`
environment your terminal session already has active).

**Why background + log-tail, not a blocking cell:** training + generation + judging can run for hours.
A cell that runs `./run_exp_4.sh` directly blocks until it's done, and if your browser tab or SSH
tunnel drops, you lose the ability to see progress (and in some Jupyter setups, the kernel dies with
it). Instead, this notebook launches the run **detached** (`nohup ... &`), so it keeps going on the
server no matter what your client does, and gives you cells to poll it: is it still running, and what
does its log say right now.

**Starting Jupyter on the GPU node** (run this once, in your terminal, not in this notebook):
```bash
cd /SEAS/home/<your-netid>/dpo/mh-dpo-master
jupyter lab --no-browser --port=8888 --ip=0.0.0.0
```
Then from your local machine, tunnel to it and open the printed `localhost:8888/...` URL in your
browser:
```bash
ssh -L 8888:<gpu-node>:8888 <your-netid>@pegasus.arc.gwu.edu
```
(If Pegasus requires going through a separate login node to reach `<gpu-node>`, or GWU already provides a
JupyterHub for HPC jobs, use whichever route actually reaches the GPU node — ask your HPC docs/admin
if this direct tunnel doesn't connect.)

## 0. Sanity checks
Confirm you're in the right place, with GPU access, before launching anything long.

In [ ]:
import subprocess, os

def sh(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True)

sh("pwd")
sh("ls run_exp_4.sh src/judge.py src/analyze.py")  # the 3 files we edited -- confirms you're in mh-dpo-master
sh("echo VIRTUAL_ENV=$VIRTUAL_ENV")
sh("python3 -c \"import torch, transformers, peft; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())\"")
sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv")

## 1. Configure the run

Edit these three values, then run the cell. This mirrors the command from our earlier discussion:
finegrained skill weights (measured `ErrorRate_base(k)`, from `results/ex4-baseline-errorrate/skill_error_rates.md`)
and the new 4-principle judge, run under a fresh `RUN_TAG` so it never overwrites an earlier run's data/model/results.

In [ ]:
RUN_TAG = "finegrained-4principle"
SKILL_WEIGHTS = '{"SQ": 1.041, "EV": 1.011, "CR": 1.0}'
JUDGE_MODEL = "NousResearch/Meta-Llama-3.1-8B-Instruct"

LOG_FILE = f"logs/exp_4-{RUN_TAG}.log"      # written by run_exp_4.sh itself
NOHUP_LOG = f"logs/nohup-{RUN_TAG}.log"     # captures anything printed outside that (e.g. a crash before logging starts)
print("This run's own log:", LOG_FILE)
print("Launch/crash log:  ", NOHUP_LOG)

## 2. Launch it in the background

`nohup ... &` detaches the process from this notebook's kernel, so it survives a dropped connection.
`disown` removes it from the shell's job table too, for the same reason. We capture its PID so the
next cells can check on it specifically (not just "is *some* run_exp_4.sh running").

In [ ]:
import subprocess, os, shlex

env = os.environ.copy()
env["RUN_TAG"] = RUN_TAG
env["SKILL_WEIGHTS"] = SKILL_WEIGHTS
env["JUDGE_MODEL"] = JUDGE_MODEL

os.makedirs("logs", exist_ok=True)

# Launch detached: stdin closed, stdout/stderr to NOHUP_LOG, new session so it isn't tied to this kernel.
with open(NOHUP_LOG, "w") as logf:
    proc = subprocess.Popen(
        ["nohup", "bash", "run_exp_4.sh"],
        stdin=subprocess.DEVNULL, stdout=logf, stderr=subprocess.STDOUT,
        env=env, start_new_session=True,
    )

with open(f".run_{RUN_TAG}.pid", "w") as f:
    f.write(str(proc.pid))

print(f"Launched run_exp_4.sh in the background, PID={proc.pid}")
print(f"PID saved to .run_{RUN_TAG}.pid -- re-run the check cells below any time, even after restarting this kernel.")

## 3. Check on it
Re-run this cell any time (even in a brand-new kernel / after closing and reopening the notebook) to see if it's still going.

In [ ]:
import subprocess

def is_running(pid):
    return subprocess.run(f"kill -0 {pid}", shell=True).returncode == 0

try:
    with open(f".run_{RUN_TAG}.pid") as f:
        pid = int(f.read().strip())
    print(f"PID {pid}: {'RUNNING' if is_running(pid) else 'not running (finished or was killed)'}")
except FileNotFoundError:
    print("No PID file yet -- run the launch cell first.")

subprocess.run(f"ps -o pid,etime,cmd -p {pid}" if 'pid' in dir() else "true", shell=True)

## 4. Tail the log
Shows the script's own step-by-step log (`== 1/7 ==`, `== 2/7 ==`, ...). Re-run to refresh.

In [ ]:
import subprocess
print(f"--- last 60 lines of {LOG_FILE} ---")
subprocess.run(f"tail -n 60 {LOG_FILE} 2>/dev/null || echo '(not written yet)'", shell=True)
print(f"\n--- last 20 lines of {NOHUP_LOG} (launch-time errors, if any) ---")
subprocess.run(f"tail -n 20 {NOHUP_LOG} 2>/dev/null || echo '(empty)'", shell=True)

## 5. GPU usage while it runs
Useful to confirm training is actually using the GPU, not silently falling back to CPU.

In [ ]:
import subprocess
subprocess.run("nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv", shell=True)

## 6. When it's done: read the summary

`run_exp_4.sh` prints `Done. Summary written to ...` as its last log line once all 7 steps finish.
Once you see that in the log (cell 4), read the actual result here.

In [ ]:
import subprocess
summary = f"results/ex4-cbtdp-dpo-{RUN_TAG}/summary_shifaa.md"
print(f"--- {summary} ---\n")
subprocess.run(f"cat {summary}", shell=True)

## Notes

- **If the kernel or browser disconnects**, the training job keeps running on the server (that's the
  point of `nohup` + `start_new_session=True`). Reopen this notebook later, re-run cell 1 with the
  *same* `RUN_TAG`, then run cells 3/4/5/6 to check on or read the result of the same run — no need to
  relaunch.
- **To stop a run**: `!kill <pid>` (from the PID printed in cell 2, or read from `.run_<RUN_TAG>.pid`).
- **Running a different config** (new skill weights, new judge model, a fresh `RUN_TAG`): just edit
  cell 1 and re-run from there. Each `RUN_TAG` gets its own log, model directory and results directory,
  so nothing from an earlier run is overwritten.
- **This is a convenience wrapper only** — it runs the exact same `run_exp_4.sh` you'd run by hand from
  the terminal. It changes nothing about the training/eval logic.